In [ ]:
%matplotlib inline

# NOTE: %reset wipes every variable in the current kernel session, and normally
# prompts "Proceed (y/[n])?" interactively when run in a live notebook - kept
# here to ensure a clean run, but will hang if this notebook is ever run
# non-interactively (e.g. via jupyter nbconvert --execute).
%reset

import h5py
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import os
import pandas as pd
import pickle
import plotly.graph_objects as go
import trimesh

from pathlib import Path
from collections import defaultdict
from scipy.io import loadmat
from scipy.spatial import cKDTree
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

projectRoot = Path().resolve().parent

# ---- USER SETTINGS ----
# Root of the drive/mount holding the MRI/atlas/mesh data (the same `driveName`
# used throughout the imageReconstruction MATLAB repo). Edit for your machine —
# pathlib.Path handles the rest cross-platform, no OS-specific branching needed.
dataDriveRoot = Path('/Volumes/YourDriveName')  # <-- edit: external drive mount point, or a local/mapped folder


In [ ]:
#User defined params
preProcPipeline = 'Standard' #'Standard, '015LPF, '025LPF'...

trialsIncluded = 'Full' # 'Hab', 'Nov, or 'Full'
firstTrial = 1
if trialsIncluded == 'Hab':
    lastTrial = 15
elif trialsIncluded == 'Nov':
    lastTrial = 20
else:
    lastTrial = 25

ttestSignificance = 0.05
cptSignificance = 0.05
# multiple comparison correction methods - e.g. 'b' = Bonferroni
ttestCorrectMethod = 'holm'
withinChanCorrectMethod = 'h' #'h' Holm
numBasis = 21
firstSamp = -40
lastSamp = 190

stimOnset  = 40
peakOnset  = 110
peakOffset = 170

parcels = 'ALL' #'ROI' for ROI channels base on previous analysis; 'ALL' for all array channels
useLongRun = False #see autocorrelation test and ACF figures below!
lrCovLag = 0 #set to 0 to calculate h based on dataset size
useWildBinSeg = True

lineStyles = [ '-', '--', '-.', ':', 'solid', 'dashed', 'dashdot', 'dotted'] # for plotting

#for indexing later on
firstFunc = firstTrial-1
lastFunc = lastTrial-1


In [ ]:
# plotting parameters

mpl.rcParams.update({
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'legend.title_fontsize': 13,
})

# colourblind-friendly palette
colourblindPalette20 = [
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#009E73",  # green
    "#F0E442",  # yellow
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#CC79A7",  # pink
    "#000000",  # black
    "#88CCEE",  # cyan
    "#44AA99",  # teal
    "#117733",  # dark green
    "#332288",  # dark blue/indigo
    "#DDCC77",  # sand
    "#999933",  # olive
    "#CC6677",  # rose
    "#882255",  # wine
    "#AA4499",  # purple
    "#DDDDDD",  # light grey
    "#771155",  # dark wine
    "#6699CC",  # light blue
]


In [ ]:
# Input filenames
parcelFilenameMetadata = projectRoot / 'dataRaw' / f'indigoSurfParcelMetadata{preProcPipeline}.csv'
parcelFilenameTimeseries = projectRoot / 'dataRaw' / f'indigoSurfParcelTrialsVlen{preProcPipeline}.h5'

# Output filenames
outputFilenameChangepoint = projectRoot / 'data' / f'indigoGroupSurfParcelChangepointDf{preProcPipeline}{trialsIncluded}.csv'
outputFilenameCoeffs = projectRoot / 'data' / f'indigoGroupSurfParcelCoeffs{preProcPipeline}{trialsIncluded}.csv'
outputFilenameParcelBothChroms = projectRoot / 'data' / f'indigoGroupSurfParcelBothChroms{preProcPipeline}{trialsIncluded}.csv'

# Directory for saved t-test result tables and other outputs (previously a separate
# hardcoded 'DATA' string defined later in the notebook - consolidated here with
# everything else so all output paths are set in one place, up front).
tablesDir = projectRoot / 'data'
tablesDir.mkdir(parents=True, exist_ok=True)
DATA = str(tablesDir) + os.sep  # kept for compatibility with str-concatenation save calls below

# ---- Age-specific mesh/atlas paths, used by the mesh visualisation section ----
agePaths = {
    '01mo': {
        'atlas': dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '01mo' / '01mo_WangFunctional_Reg_Head.nii.gz',
        'gmvol': dataDriveRoot / 'mri' / 'registered' / 'UNC_to_NeuroDev' / 'No Mask' / '01mo' / '01mo_GM_Reg_Head.nii.gz',
        'mat':   dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface01mo.mat',
    },
    '06mo': {
        'atlas': dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '06mo' / '06mo_WangFunctional_Reg_Head.nii.gz',
        'gmvol': dataDriveRoot / 'mri' / 'registered' / 'UNC_to_NeuroDev' / 'No Mask' / '06mo' / '06mo_GM_Reg_Head.nii.gz',
        'mat':   dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface06mo.mat',
    },
    '12mo': {
        'atlas': dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '12mo' / '12mo_WangFunctional_Reg_Head.nii.gz',
        'gmvol': dataDriveRoot / 'mri' / 'registered' / 'UNC_to_NeuroDev' / 'No Mask' / '12mo' / '12mo_GM_Reg_Head.nii.gz',
        'mat':   dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface12mo.mat',
    },
}
# Adjust this mapping if your Age values are integers (1, 6, 12) or strings ('01mo' etc.)
ageKeyMap = {1: '01mo', 6: '06mo', 12: '12mo'}


In [ ]:
# Load retention CSVs and build a set of valid (age, infant) pairs
retentionFiles = {
    1: projectRoot / 'dataRaw' / 'DSMB_nirs_data_retention_1mo.csv',
    6: projectRoot / 'dataRaw' / 'DSMB_nirs_data_retention_6mo.csv',
    12: projectRoot / 'dataRaw' / 'DSMB_nirs_data_retention_12mo.csv'
}

validInfants = set()
for age, filepath in retentionFiles.items():
    retDf = pd.read_csv(filepath, encoding='utf-8-sig')
    retDf = retDf.dropna(subset=['indigo_ID'])
    retDf['Infant'] = retDf['indigo_ID'].str.extract(r'(\d{3})')[0].astype(int)
    included = retDf[retDf['included'] == 1]['Infant']
    for infant in included:
        validInfants.add((age, infant))


In [ ]:
# Load metadata CSV
metadataDf = pd.read_csv(parcelFilenameMetadata)


In [ ]:
metadataDf = metadataDf[
    metadataDf.apply(lambda row: (row['Age'], row['Infant']) in validInfants, axis=1)
]


In [ ]:
with h5py.File(parcelFilenameTimeseries, 'r') as h5f:
    tsDataset = h5f['timeSeries'][()].copy()


Data visualisation as first check

In [ ]:
# uniqueAges = metadataDf['Age'].unique()

# for age in uniqueAges:

#     # Parcel data has no pre-stimulus baseline; onset is at sample 0

#     subsetDf = metadataDf[metadataDf['Age'] == age]
#     uniqueCombos = subsetDf[['Parcel', 'Chrom']].drop_duplicates().values

#     plt.figure(figsize=(12, 6))
#     plt.title(f"Age {age}: Mean time series per Parcel-Chromophore combination")

#     for combo in uniqueCombos:
#         parcel, chrom = combo

#         comboDf = subsetDf[
#             (subsetDf['Parcel'] == parcel) &
#             (subsetDf['Chrom'] == chrom)
#         ]

#         tsList = [tsDataset[tsIDx]*1000000 for tsIDx in comboDf['tsIDx']]

#         if len(tsList) == 0:
#             continue

#         minLen = min(len(ts) for ts in tsList)
#         truncatedTs = np.array([ts[:minLen] for ts in tsList])

#         meanTs = np.mean(truncatedTs, axis=0)
#         onsetCentredTs = meanTs - meanTs[40]

#         color = colourblindPalette20[parcel % len(colourblindPalette20)]
#         linestyle = '--' if chrom == 2 else '-'

#         plt.plot(np.arange(len(onsetCentredTs)), onsetCentredTs,
#                     label=f"Parcel {parcel} - Chrom {chrom}",
#                     linestyle=linestyle, alpha=0.8)

# plt.xlabel("Time (0.1s)")
# plt.ylabel("Chromophore concentration change")
# plt.tight_layout()
# plt.show()


Significance testing using t-tests

In [ ]:
results = []

for age in sorted(metadataDf['Age'].unique()):

    ageDf = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Trial'] <= 5)]
    grouped = ageDf.groupby(['Parcel', 'Chrom', 'Infant'])

    comboToInfants = defaultdict(list)
    for (parcel, chrom, infant), group in grouped:
        comboToInfants[(parcel, chrom)].append((infant, group))

    for (parcel, chrom), infantGroups in comboToInfants.items():

        infantPeaks = []
        infantBaselines = []

        for infant, group in infantGroups:
            trialPeaks = []
            trialBaselines = []

            for idx in group['tsIDx']:
                ts = tsDataset[idx] * 1000000
                if np.isnan(ts).all():
                    continue
                trialPeaks.append(np.nanmean(ts[peakOnset:peakOffset]))
                trialBaselines.append(np.nanmean(ts[:stimOnset]))

            if trialPeaks:
                infantPeaks.append(np.mean(trialPeaks))
                infantBaselines.append(np.mean(trialBaselines))

        if len(infantPeaks) >= 5:

            tVal, pVal = ttest_rel(infantPeaks, infantBaselines)
            # Two-tailed: any significant change from baseline
            results.append({
                'Age': age, 'Parcel': parcel,
                'Chrom': chrom,
                'Mean Peak': np.mean(infantPeaks),
                'SD Peak': np.std(infantPeaks),
                't': tVal, 'p': pVal, 'n': len(infantPeaks)
            })

resultsDf = pd.DataFrame(results)
resultsDf['Adjusted p-value'] = multipletests(resultsDf['p'], method=ttestCorrectMethod)[1]
resultsDf['Significant'] = resultsDf['Adjusted p-value'] < 0.05
resultsDf.to_csv(tablesDir / "surf_parcel_t_test_results_corrected.csv", index=False)
print(resultsDf.head())


In [ ]:
adjustedResultsDf = resultsDf[resultsDf['Adjusted p-value'] < 0.05]
adjustedResultsDf.to_csv(tablesDir / "surf_parcel_t_test_results_corrected_SIGONLY.csv", index=False)
print("Activation")
print(adjustedResultsDf.head())


In [ ]:
peakSign = {
    (row['Age'], row['Parcel'], row['Chrom']): np.sign(row['Mean Peak'])
    for _, row in adjustedResultsDf.iterrows()
}


In [ ]:
validCombos = set(
    tuple(row) for row in adjustedResultsDf[['Parcel', 'Chrom']].values
)
validAgeCombos = set(
    tuple(row) for row in adjustedResultsDf[['Age', 'Parcel', 'Chrom']].values
)

uniqueAges = sorted(metadataDf['Age'].unique())

# --- Pass 1: collect global y limits ---
globalMin, globalMax = np.inf, -np.inf

for age in uniqueAges:
    subsetDf = metadataDf[metadataDf['Age'] == age]
    uniqueCombos = subsetDf[['Parcel', 'Chrom']].drop_duplicates().values
    for combo in uniqueCombos:
        parcel, chrom = combo
        if (parcel, chrom) not in validCombos:
            continue
        comboDf = subsetDf[(subsetDf['Parcel'] == parcel) & (subsetDf['Chrom'] == chrom)]
        tsList = [tsDataset[tsIDx]*1000000 for tsIDx in comboDf['tsIDx']]
        if len(tsList) == 0:
            continue
        minLen = min(len(ts) for ts in tsList)
        truncatedTs = np.array([ts[:minLen] for ts in tsList])
        meanTs = np.mean(truncatedTs, axis=0)
        onsetCentredTs = meanTs - meanTs[40]
        globalMin = min(globalMin, np.min(onsetCentredTs))
        globalMax = max(globalMax, np.max(onsetCentredTs))

yRange = globalMax - globalMin
yMin = globalMin - 0.1 * yRange
yMax = globalMax + 0.1 * yRange

# --- Pass 2: plot ---
for age in uniqueAges:
    subsetDf = metadataDf[metadataDf['Age'] == age]
    uniqueCombos = subsetDf[['Parcel', 'Chrom']].drop_duplicates().values

    plt.figure(figsize=(12, 6))
    plt.title(f"Age {age}: Block averaged responses for significant Parcels")

    for combo in uniqueCombos:
        parcel, chrom = combo
        if (parcel, chrom) not in validCombos:
            continue

        comboDf = subsetDf[(subsetDf['Parcel'] == parcel) & (subsetDf['Chrom'] == chrom)]
        tsList = [tsDataset[tsIDx]*1000000 for tsIDx in comboDf['tsIDx']]
        if len(tsList) == 0:
            continue

        minLen = min(len(ts) for ts in tsList)
        truncatedTs = np.array([ts[:minLen] for ts in tsList])
        meanTs = np.mean(truncatedTs, axis=0)
        onsetCentredTs = meanTs - meanTs[40]

        colour = colourblindPalette20[parcel % len(colourblindPalette20)]
        lineWidth = 3 if (age, parcel, chrom) in validAgeCombos else 0.7
        linestyle = (
            '-' if (age, parcel, chrom) in validAgeCombos and chrom == 1 else
            '-.' if (age, parcel, chrom) in validAgeCombos and chrom == 2 else
            ':'
        )

        label = str(parcel) if (age, parcel, chrom) in validAgeCombos else None

        plt.plot(
            np.arange(-(stimOnset), len(onsetCentredTs) - stimOnset),
            onsetCentredTs,
            label=label,
            color=colour,
            linestyle=linestyle,
            alpha=0.8,
            linewidth=lineWidth
        )
        if (age, parcel, chrom) in validAgeCombos:
            semTs = np.std(truncatedTs, axis=0, ddof=1) / np.sqrt(len(truncatedTs))
            onsetCentredSem = semTs
            plt.fill_between(np.arange(-(stimOnset), len(onsetCentredTs)-stimOnset), onsetCentredTs - onsetCentredSem, onsetCentredTs + onsetCentredSem, color=colour, alpha=0.2)

    plt.axvspan(0, 90, color='lightgrey', alpha=0.3, zorder=0)
    plt.axhline(y=0, color='black', linewidth=1.2, linestyle='--', alpha=0.8)

    plt.xlabel("Time (0.1s)", fontsize=14)
    plt.ylabel("Chromophore concentration change (µM)", fontsize=14)
    plt.tight_layout()
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(
        by_label.values(),
        by_label.keys(),
        title="Parcel",
        facecolor='white',
        framealpha=1,
        loc='upper left',
        borderaxespad=1,
        ncol=min(len(by_label), 5),
    )
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.ylim(yMin, yMax)
    plt.xlim(-40, 190)
    plt.show()


In [ ]:
validCombos = set(
    tuple(row) for row in adjustedResultsDf[['Parcel', 'Chrom']].values
)
validAgeCombos = set(
    tuple(row) for row in adjustedResultsDf[['Age', 'Parcel', 'Chrom']].values
)

# Parcels significant in both chromophores at a given age
bothChromsCombos = set()
for (a, p, c) in validAgeCombos:
    if (a, p, 1) in validAgeCombos and (a, p, 2) in validAgeCombos:
        bothChromsCombos.add((a, p))

uniqueAges = sorted(metadataDf['Age'].unique())

# --- Pass 1: collect global y limits ---
globalMin, globalMax = np.inf, -np.inf
for age in uniqueAges:
    subsetDf = metadataDf[metadataDf['Age'] == age]
    uniqueCombos = subsetDf[['Parcel', 'Chrom']].drop_duplicates().values
    for combo in uniqueCombos:
        parcel, chrom = combo
        if (parcel, chrom) not in validCombos:
            continue
        comboDf = subsetDf[(subsetDf['Parcel'] == parcel) & (subsetDf['Chrom'] == chrom)]
        tsList = [tsDataset[tsIDx]*1000000 for tsIDx in comboDf['tsIDx']]
        if len(tsList) == 0:
            continue
        minLen = min(len(ts) for ts in tsList)
        truncatedTs = np.array([ts[:minLen] for ts in tsList])
        meanTs = np.mean(truncatedTs, axis=0)
        onsetCentredTs = meanTs - meanTs[40]
        globalMin = min(globalMin, np.min(onsetCentredTs))
        globalMax = max(globalMax, np.max(onsetCentredTs))

yRange = globalMax - globalMin
yMin = globalMin - 0.1 * yRange
yMax = globalMax + 0.1 * yRange

# --- Pass 2: plot ---
for age in uniqueAges:
    subsetDf = metadataDf[metadataDf['Age'] == age]
    uniqueCombos = subsetDf[['Parcel', 'Chrom']].drop_duplicates().values
    plt.figure(figsize=(12, 6))
    plt.title(f"{age}mo: group averaged responses for significant parcels")
    for combo in uniqueCombos:
        parcel, chrom = combo
        if (parcel, chrom) not in validCombos:
            continue
        comboDf = subsetDf[(subsetDf['Parcel'] == parcel) & (subsetDf['Chrom'] == chrom)]
        tsList = [tsDataset[tsIDx]*1000000 for tsIDx in comboDf['tsIDx']]
        if len(tsList) == 0:
            continue
        minLen = min(len(ts) for ts in tsList)
        truncatedTs = np.array([ts[:minLen] for ts in tsList])
        meanTs = np.mean(truncatedTs, axis=0)
        onsetCentredTs = meanTs - meanTs[40]
        colour = 'red' if chrom == 1 else 'blue'
        lineWidth = 3 if (age, parcel, chrom) in validAgeCombos else 0.7
        linestyle = (
            '-' if (age, parcel, chrom) in validAgeCombos and (age, parcel) in bothChromsCombos else
            '-.' if (age, parcel, chrom) in validAgeCombos else
            ':'
        )
        label = str(parcel) if (age, parcel, chrom) in validAgeCombos else None
        plt.plot(
            np.arange(-(stimOnset), len(onsetCentredTs) - stimOnset),
            onsetCentredTs,
            label=label,
            color=colour,
            linestyle=linestyle,
            alpha=0.8,
            linewidth=lineWidth
        )
        if (age, parcel, chrom) in validAgeCombos:
            semTs = np.std(truncatedTs, axis=0, ddof=1) / np.sqrt(len(truncatedTs))
            onsetCentredSem = semTs
            plt.fill_between(
                np.arange(-(stimOnset), len(onsetCentredTs) - stimOnset),
                onsetCentredTs - onsetCentredSem,
                onsetCentredTs + onsetCentredSem,
                color=colour,
                alpha=0.15
            )
    plt.axvspan(0, 90, color='lightgrey', alpha=0.3, zorder=0)
    plt.axhline(y=0, color='black', linewidth=1.2, linestyle='--', alpha=0.8)
    plt.xlabel("Time (0.1s)", fontsize=14)
    plt.ylabel("Chromophore concentration change (µM)", fontsize=14)
    plt.tight_layout()
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))

    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.ylim(yMin, yMax)
    plt.xlim(-40, 190)
    plt.show()


Filter for significant results in both chroms:

Test significant parcels for habituation i.e. Hab trials 1-5 v Hab trials 11-15

In [ ]:
results = []

if 'adjustedResultsDf' in locals() and not adjustedResultsDf.empty:

    sigCombos = adjustedResultsDf[['Age', 'Parcel', 'Chrom']].drop_duplicates()
    sigMetadataDf = metadataDf.merge(sigCombos, on=['Age', 'Parcel', 'Chrom'])

    for age in sorted(sigMetadataDf['Age'].unique()):

        hab1Df = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] <= 5)]
        hab3Df = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] > 10) & (sigMetadataDf['Trial'] <= 15)]

        hab1Grouped = hab1Df.groupby(['Parcel', 'Chrom', 'Infant'])
        hab3Grouped = hab3Df.groupby(['Parcel', 'Chrom', 'Infant'])

        # Build nested dictionaries for fast lookups
        hab1ComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in hab1Grouped:
            hab1ComboToInfants[(parcel, chrom)].append((infant, group))
        hab3ComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in hab3Grouped:
            hab3ComboToInfants[(parcel, chrom)].append((infant, group))

        # find common combinations and infants
        commonCombos = set(hab1ComboToInfants.keys()) & set(hab3ComboToInfants.keys())

        for (parcel, chrom) in commonCombos:

            hab1Infants = hab1ComboToInfants[(parcel, chrom)]
            hab3Infants = hab3ComboToInfants[(parcel, chrom)]

            hab1InfantIDs = {infant for (infant, _) in hab1Infants}
            hab3InfantIDs = {infant for (infant, _) in hab3Infants}
            commonInfants = list(hab1InfantIDs & hab3InfantIDs)

            hab1Peaks = []
            hab3Peaks = []

            hab1Dict = dict(hab1Infants)
            hab3Dict = dict(hab3Infants)

            for infant in commonInfants:

                hab1Group = hab1Dict[infant]
                hab3Group = hab3Dict[infant]

                infantHab1Peaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in hab1Group['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]
                infantHab3Peaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in hab3Group['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]

                if infantHab1Peaks and infantHab3Peaks:
                    if len(infantHab1Peaks) > 2 and len(infantHab3Peaks) > 2:
                        hab1Peaks.append(np.mean(infantHab1Peaks))
                        hab3Peaks.append(np.mean(infantHab3Peaks))
            if len(hab1Peaks) >= 5:
                tVal, pVal = ttest_rel(hab1Peaks, hab3Peaks)
                sign = peakSign.get((age, parcel, chrom), 1)
                pVal = pVal / 2 if np.sign(tVal) == sign else 1.0
                if np.isnan(pVal):
                    print(f"NaN p-val: t={tVal}, hab1Peaks={hab1Peaks}, hab3Peaks={hab3Peaks}")
                results.append({
                    'Age': age, 'Parcel': parcel,
                    'Chrom': chrom, 'Mean Hab 1': np.mean(hab1Peaks), 'SD Hab1': np.std(hab1Peaks),
                    'Mean Hab3': np.mean(hab3Peaks), 'SD Hab3': np.std(hab3Peaks), 't': tVal, 'p': pVal, 'n': len(hab1Peaks)
                })

    habDf = pd.DataFrame(results)
    habDf['Adjusted p-value'] = multipletests(habDf['p'], method=ttestCorrectMethod)[1]
    habDf['Significant'] = habDf['Adjusted p-value'] < 0.05
    habDf.to_csv(tablesDir / "surf_parcel_t_test_hab_corrected.csv", index=False)
    print(habDf)


In [ ]:
if 'habDf' in locals() and not habDf.empty:

    adjustedHabDf = habDf[habDf['Adjusted p-value'] < 0.05]
    adjustedHabDf.to_csv(tablesDir / "surf_parcel_t_test_hab_corrected_SIGONLY.csv", index=False)
    print("Habituation")
    print(adjustedHabDf)


In [ ]:
# Define a standard length for time series (change this if needed)
targetLen = 231

def pad_or_truncate(ts, target_len):
    if len(ts) > target_len:
        return ts[:target_len]
    elif len(ts) < target_len:
        return np.pad(ts, (0, target_len - len(ts)), mode='constant', constant_values=np.nan)
    else:
        return ts

for _, row in adjustedHabDf.iterrows():
    age = row['Age']
    parcel = row['Parcel']
    chrom = row['Chrom']

    plotStimOnset, plotPeakOnset, plotPeakOffset = (0, 80, 120)

    hab1Df = metadataDf[
        (metadataDf['Age'] == age) &
        (metadataDf['Parcel'] == parcel) &
        (metadataDf['Chrom'] == chrom) &
        (metadataDf['Trial'] <= 5)
    ]
    hab3Df = metadataDf[
        (metadataDf['Age'] == age) &
        (metadataDf['Parcel'] == parcel) &
        (metadataDf['Chrom'] == chrom) &
        (metadataDf['Trial'] > 10) &
        (metadataDf['Trial'] <= 15)
    ]

    hab1Grouped = hab1Df.groupby('Infant')
    hab3Grouped = hab3Df.groupby('Infant')

    hab1Dict = {infant: group for infant, group in hab1Grouped}
    hab3Dict = {infant: group for infant, group in hab3Grouped}

    commonInfants = set(hab1Dict.keys()) & set(hab3Dict.keys())

    hab1Curves = []
    hab3Curves = []

    for infant in commonInfants:
        ts1 = [
            pad_or_truncate(tsDataset[idx] * 1e6, targetLen)
            for idx in hab1Dict[infant]['tsIDx']
            if not np.isnan(tsDataset[idx]).all()
        ]
        ts3 = [
            pad_or_truncate(tsDataset[idx] * 1e6, targetLen)
            for idx in hab3Dict[infant]['tsIDx']
            if not np.isnan(tsDataset[idx]).all()
        ]

        if len(ts1) > 0 and len(ts3) > 0:
            mean_ts1 = np.nanmean(ts1, axis=0)
            mean_ts3 = np.nanmean(ts3, axis=0)
            hab1Curves.append(mean_ts1)
            hab3Curves.append(mean_ts3)

    if hab1Curves and hab3Curves:
        hab1Curves = np.vstack(hab1Curves)
        hab3Curves = np.vstack(hab3Curves)

        mean_hab1 = np.nanmean(hab1Curves, axis=0)
        sem_hab1 = np.nanstd(hab1Curves, axis=0) / np.sqrt(hab1Curves.shape[0])

        mean_hab3 = np.nanmean(hab3Curves, axis=0)
        sem_hab3 = np.nanstd(hab3Curves, axis=0) / np.sqrt(hab3Curves.shape[0])

        # Create time axis relative to stimulus onset
        timeAxis = np.arange(-plotStimOnset, targetLen - plotStimOnset)

        # Colour by chrom
        colour = 'red' if chrom == 1 else 'blue'

        # Plot
        plt.figure(figsize=(8, 4))
        plt.plot(timeAxis, mean_hab1, label='Fam1', linewidth=2, linestyle='solid', color=colour)
        plt.fill_between(timeAxis, mean_hab1 - sem_hab1, mean_hab1 + sem_hab1, alpha=0.3, color=colour)

        plt.plot(timeAxis, mean_hab3, label='Nov', linewidth=2, linestyle='dashed', color=colour)
        plt.fill_between(timeAxis, mean_hab3 - sem_hab3, mean_hab3 + sem_hab3, alpha=0.3, color=colour)

        plt.axvspan(0, 90, color='lightgrey', alpha=0.3, zorder=0)

        plt.title(f"Age {age} | Parcel {parcel} | Chrom {chrom}")
        plt.xlabel("Time from stimulus onset (samples)")
        plt.ylabel("Hb concentration (μM)")
        plt.legend()
        plt.tight_layout()
        ax = plt.gca()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.xlim(-40, 190)
        plt.show()


Test for novelty detection

In [ ]:
results = []

if 'adjustedHabDf' in locals() and not adjustedHabDf.empty:

    habCombos = adjustedHabDf[['Age', 'Parcel', 'Chrom']].drop_duplicates()

    habMetadataDf = metadataDf.merge(habCombos, on=['Age', 'Parcel', 'Chrom'])

    for age in sorted(habMetadataDf['Age'].unique()):

        hab3Df = habMetadataDf[(habMetadataDf['Age'] == age) & (habMetadataDf['Trial'] > 10) & (habMetadataDf['Trial'] <= 15)]
        novDf = habMetadataDf[(habMetadataDf['Age'] == age) & (habMetadataDf['Trial'] > 15) & (habMetadataDf['Trial'] <= 20)]

        hab3Grouped = hab3Df.groupby(['Parcel', 'Chrom', 'Infant'])
        novGrouped = novDf.groupby(['Parcel', 'Chrom', 'Infant'])

        # Build nested dictionaries for fast lookups
        hab3ComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in hab3Grouped:
            hab3ComboToInfants[(parcel, chrom)].append((infant, group))
        novComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in novGrouped:
            novComboToInfants[(parcel, chrom)].append((infant, group))

        # find common combinations and infants
        commonCombos = set(novComboToInfants.keys()) & set(hab3ComboToInfants.keys())

        for (parcel, chrom) in commonCombos:

            novInfants = novComboToInfants[(parcel, chrom)]
            hab3Infants = hab3ComboToInfants[(parcel, chrom)]

            novInfantIDs = {infant for (infant, _) in novInfants}
            hab3InfantIDs = {infant for (infant, _) in hab3Infants}
            commonInfants = list(novInfantIDs & hab3InfantIDs)

            novPeaks = []
            hab3Peaks = []

            novDict = dict(novInfants)
            hab3Dict = dict(hab3Infants)

            for infant in commonInfants:

                novGroup = novDict[infant]
                hab3Group = hab3Dict[infant]

                infantNovPeaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in novGroup['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]
                infantHab3Peaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in hab3Group['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]

                if infantNovPeaks and infantHab3Peaks:
                    if len(infantNovPeaks) > 2 and len(infantHab3Peaks) > 2:
                        novPeaks.append(np.mean(infantNovPeaks))
                        hab3Peaks.append(np.mean(infantHab3Peaks))
            if len(novPeaks) >= 5:
                tVal, pVal = ttest_rel(novPeaks, hab3Peaks)
                sign = peakSign.get((age, parcel, chrom), 1)
                # Nov should recover toward original activation direction
                pVal = pVal / 2 if np.sign(tVal) == sign else 1.0
                if np.isnan(pVal):
                    print(f"NaN p-val: t={tVal}, novPeaks={novPeaks}, hab3Peaks={hab3Peaks}")
                results.append({
                    'Age': age, 'Parcel': parcel,
                    'Chrom': chrom, 'Mean Hab 3': np.mean(hab3Peaks), 'SD Hab3': np.std(hab3Peaks),
                    'Mean Nov': np.mean(novPeaks), 'SD Nov': np.std(novPeaks), 't': tVal, 'p': pVal, 'n': len(novPeaks)
                })



    noveltyDf = pd.DataFrame(results)
    noveltyDf['Adjusted p-value'] = multipletests(noveltyDf['p'], method=ttestCorrectMethod)[1]
    noveltyDf['Significant'] = noveltyDf['Adjusted p-value'] < 0.05
    noveltyDf.to_csv(tablesDir / "surf_parcel_t_test_nov_corrected.csv", index=False)
    print(noveltyDf)


In [ ]:
if 'noveltyDf' in locals() and not noveltyDf.empty:

    adjustedNoveltyDf = noveltyDf[noveltyDf['Adjusted p-value'] < 0.05]
    adjustedNoveltyDf.to_csv(tablesDir / "surf_parcel_t_test_nov_corrected_SIGONLY.csv", index=False)
    print("Novelty Response")
    print(adjustedNoveltyDf)


In [ ]:
results = []

if 'adjustedHabDf' in locals() and not adjustedHabDf.empty:

    sigCombos = adjustedHabDf[['Age', 'Parcel', 'Chrom']].drop_duplicates()
    sigMetadataDf = metadataDf.merge(sigCombos, on=['Age', 'Parcel', 'Chrom'])

    for age in sorted(sigMetadataDf['Age'].unique()):

        hab3Df = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] > 10) & (sigMetadataDf['Trial'] <= 15)]
        novDf = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] > 15) & (sigMetadataDf['Trial'] <= 20)]

        hab3Grouped = hab3Df.groupby(['Parcel', 'Chrom', 'Infant'])
        novGrouped = novDf.groupby(['Parcel', 'Chrom', 'Infant'])

        # Build nested dictionaries for fast lookups
        hab3ComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in hab3Grouped:
            hab3ComboToInfants[(parcel, chrom)].append((infant, group))
        novComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in novGrouped:
            novComboToInfants[(parcel, chrom)].append((infant, group))

        # find common combinations and infants
        commonCombos = set(novComboToInfants.keys()) & set(hab3ComboToInfants.keys())

        for (parcel, chrom) in commonCombos:

            novInfants = novComboToInfants[(parcel, chrom)]
            hab3Infants = hab3ComboToInfants[(parcel, chrom)]

            novInfantIDs = {infant for (infant, _) in novInfants}
            hab3InfantIDs = {infant for (infant, _) in hab3Infants}
            commonInfants = list(novInfantIDs & hab3InfantIDs)

            novPeaks = []
            hab3Peaks = []

            novDict = dict(novInfants)
            hab3Dict = dict(hab3Infants)

            for infant in commonInfants:

                novGroup = novDict[infant]
                hab3Group = hab3Dict[infant]

                infantNovPeaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in novGroup['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]
                infantHab3Peaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in hab3Group['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]

                if infantNovPeaks and infantHab3Peaks:
                    if len(infantNovPeaks) > 2 and len(infantHab3Peaks) > 2:
                        novPeaks.append(np.mean(infantNovPeaks))
                        hab3Peaks.append(np.mean(infantHab3Peaks))
            if len(novPeaks) >= 5:
                tVal, pVal = ttest_rel(novPeaks, hab3Peaks)
                sign = peakSign.get((age, parcel, chrom), 1)
                # Continued habituation: Nov still smaller in magnitude than Hab3
                # so novPeaks < hab3Peaks for positive activation, novPeaks > hab3Peaks for negative
                pVal = pVal / 2 if np.sign(tVal) == -sign else 1.0
                if np.isnan(pVal):
                    print(f"NaN p-val: t={tVal}, novPeaks={novPeaks}, hab3Peaks={hab3Peaks}")
                results.append({
                    'Age': age, 'Parcel': parcel,
                    'Chrom': chrom, 'Mean Hab 3': np.mean(hab3Peaks), 'SD Hab3': np.std(hab3Peaks),
                    'Mean Nov': np.mean(novPeaks), 'SD Nov': np.std(novPeaks), 't': tVal, 'p': pVal, 'n': len(novPeaks)
                })



    contHabDf = pd.DataFrame(results)
    contHabDf['Adjusted p-value'] = multipletests(contHabDf['p'], method=ttestCorrectMethod)[1]
    contHabDf['Significant'] = contHabDf['Adjusted p-value'] < 0.05
    contHabDf.to_csv(tablesDir / "surf_parcel_t_test_conthab_corrected.csv", index=False)
    print(contHabDf)


In [ ]:
if 'contHabDf' in locals() and not contHabDf.empty:

    adjustedContHabDf = contHabDf[contHabDf['Adjusted p-value'] < 0.05]
    adjustedContHabDf.to_csv(tablesDir / "surf_parcel_t_test_posthab_corrected_SIGONLY.csv", index=False)
    print(adjustedContHabDf)


Test for delayed habituation

In [ ]:
results = []

if 'adjustedResultsDf' in locals() and not adjustedResultsDf.empty:

    sigCombos = adjustedResultsDf[['Age', 'Parcel', 'Chrom']].drop_duplicates()
    sigMetadataDf = metadataDf.merge(sigCombos, on=['Age', 'Parcel', 'Chrom'])

    for age in sorted(sigMetadataDf['Age'].unique()):

        hab1Df = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] <= 5)]
        novDf = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] > 15) & (sigMetadataDf['Trial'] <= 20)]

        hab1Grouped = hab1Df.groupby(['Parcel', 'Chrom', 'Infant'])
        novGrouped = novDf.groupby(['Parcel', 'Chrom', 'Infant'])

        # Build nested dictionaries for fast lookups
        hab1ComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in hab1Grouped:
            hab1ComboToInfants[(parcel, chrom)].append((infant, group))
        novComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in novGrouped:
            novComboToInfants[(parcel, chrom)].append((infant, group))

        # find common combinations and infants
        commonCombos = set(novComboToInfants.keys()) & set(hab1ComboToInfants.keys())

        for (parcel, chrom) in commonCombos:

            novInfants = novComboToInfants[(parcel, chrom)]
            hab1Infants = hab1ComboToInfants[(parcel, chrom)]

            novInfantIDs = {infant for (infant, _) in novInfants}
            hab1InfantIDs = {infant for (infant, _) in hab1Infants}
            commonInfants = list(novInfantIDs & hab1InfantIDs)

            novPeaks = []
            hab1Peaks = []

            novDict = dict(novInfants)
            hab1Dict = dict(hab1Infants)

            for infant in commonInfants:

                novGroup = novDict[infant]
                hab1Group = hab1Dict[infant]

                infantNovPeaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in novGroup['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]
                infantHab1Peaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in hab1Group['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]

                if infantNovPeaks and infantHab1Peaks:
                    if len(infantNovPeaks) > 2 and len(infantHab1Peaks) > 2:
                        novPeaks.append(np.mean(infantNovPeaks))
                        hab1Peaks.append(np.mean(infantHab1Peaks))
            if len(novPeaks) >= 5:
                tVal, pVal = ttest_rel(novPeaks, hab1Peaks)
                sign = peakSign.get((age, parcel, chrom), 1)
                # Nov should be smaller in magnitude than Fam1, so opposite sign to activation
                pVal = pVal / 2 if np.sign(tVal) == -sign else 1.0
                if np.isnan(pVal):
                    print(f"NaN p-val: t={tVal}, novPeaks={novPeaks}, hab1Peaks={hab1Peaks}")
                results.append({
                    'Age': age, 'Parcel': parcel,
                    'Chrom': chrom, 'Mean Hab 1': np.mean(hab1Peaks), 'SD Hab1': np.std(hab1Peaks),
                    'Mean Nov': np.mean(novPeaks), 'SD Nov': np.std(novPeaks), 't': tVal, 'p': pVal, 'n': len(novPeaks)
                })



    delHabDf = pd.DataFrame(results)
    delHabDf['Adjusted p-value'] = multipletests(delHabDf['p'], method=ttestCorrectMethod)[1]
    delHabDf['Significant'] = delHabDf['Adjusted p-value'] < 0.05
    delHabDf.to_csv(tablesDir / "surf_parcel_t_test_exthab_corrected.csv", index=False)
    print(delHabDf)


In [ ]:
if not delHabDf.empty:

    adjustedDelHabDf = delHabDf[delHabDf['Adjusted p-value'] < 0.05]
    adjustedDelHabDf.to_csv(tablesDir / "surf_parcel_t_test_delhab_corrected_SIGONLY.csv", index=False)
    print("Delayed Habituation")
    print(adjustedDelHabDf)


Test for delayed habituation (Hab1 v Post-test)

In [ ]:
results = []

if 'adjustedResultsDf' in locals() and not adjustedResultsDf.empty:

    sigCombos = adjustedResultsDf[['Age', 'Parcel', 'Chrom']].drop_duplicates()
    sigMetadataDf = metadataDf.merge(sigCombos, on=['Age', 'Parcel', 'Chrom'])

    for age in sorted(sigMetadataDf['Age'].unique()):

        hab1Df = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] <= 5)]
        postDf = sigMetadataDf[(sigMetadataDf['Age'] == age) & (sigMetadataDf['Trial'] > 20) & (sigMetadataDf['Trial'] <= 25)]

        hab1Grouped = hab1Df.groupby(['Parcel', 'Chrom', 'Infant'])
        postGrouped = postDf.groupby(['Parcel', 'Chrom', 'Infant'])

        # Build nested dictionaries for fast lookups
        hab1ComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in hab1Grouped:
            hab1ComboToInfants[(parcel, chrom)].append((infant, group))
        postComboToInfants = defaultdict(list)
        for (parcel, chrom, infant), group in postGrouped:
            postComboToInfants[(parcel, chrom)].append((infant, group))

        # find common combinations and infants
        commonCombos = set(postComboToInfants.keys()) & set(hab1ComboToInfants.keys())

        for (parcel, chrom) in commonCombos:

            postInfants = postComboToInfants[(parcel, chrom)]
            hab1Infants = hab1ComboToInfants[(parcel, chrom)]

            postInfantIDs = {infant for (infant, _) in postInfants}
            hab1InfantIDs = {infant for (infant, _) in hab1Infants}
            commonInfants = list(postInfantIDs & hab1InfantIDs)

            postPeaks = []
            hab1Peaks = []

            postDict = dict(postInfants)
            hab1Dict = dict(hab1Infants)

            for infant in commonInfants:

                postGroup = postDict[infant]
                hab1Group = hab1Dict[infant]

                infantPostPeaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in postGroup['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]
                infantHab1Peaks = [
                    np.nanmean((tsDataset[idx][peakOnset:peakOffset])*1000000)
                    for idx in hab1Group['tsIDx'] if not np.isnan(tsDataset[idx]).all()
                ]

                if infantPostPeaks and infantHab1Peaks:
                    if len(infantPostPeaks) > 2 and len(infantHab1Peaks) > 2:
                        postPeaks.append(np.mean(infantPostPeaks))
                        hab1Peaks.append(np.mean(infantHab1Peaks))
            if len(postPeaks) >= 5:
                tVal, pVal = ttest_rel(postPeaks, hab1Peaks)
                sign = peakSign.get((age, parcel, chrom), 1)
                # Post should be smaller in magnitude than Fam1, so opposite sign to activation
                pVal = pVal / 2 if np.sign(tVal) == -sign else 1.0
                if np.isnan(pVal):
                    print(f"NaN p-val: t={tVal}, postPeaks={postPeaks}, hab1Peaks={hab1Peaks}")
                results.append({
                    'Age': age, 'Parcel': parcel,
                    'Chrom': chrom, 'Mean Hab 1': np.mean(hab1Peaks), 'SD Hab1': np.std(hab1Peaks),
                    'Mean Post': np.mean(postPeaks), 'SD Post': np.std(postPeaks), 't': tVal, 'p': pVal, 'n': len(postPeaks)
                })



    lrHabDf = pd.DataFrame(results)
    lrHabDf['Adjusted p-value'] = multipletests(lrHabDf['p'], method=ttestCorrectMethod)[1]
    lrHabDf['Significant'] = lrHabDf['Adjusted p-value'] < 0.05
    lrHabDf.to_csv(tablesDir / "surf_parcel_t_test_posthab_corrected.csv", index=False)
    print(lrHabDf)


In [ ]:
if not lrHabDf.empty:

    adjustedLrHabDf = lrHabDf[lrHabDf['Adjusted p-value'] < 0.05]
    adjustedLrHabDf.to_csv(tablesDir / "surf_parcel_t_test_posthab_corrected_SIGONLY.csv", index=False)
    print("Long Range Habituation")
    print(adjustedLrHabDf)


## Plot meshes

Group-level TFCE section removed (not used in the paper). The mesh-plotting setup below (atlas loading, parcel-colour lookup, mesh loading per age) was previously duplicated four times across four near-identical cells — consolidated here into shared helper functions, with each of the four original visualisations kept as a short call at the end.

In [ ]:
# ---------------- Shared mesh-plotting setup ----------------

def parcelColour(parcel, age, validAgeCombos, asUint8=False):
    parcel = int(parcel)
    age = int(age)
    match = any((age, parcel, chrom) in validAgeCombos for chrom in [1, 2])
    if match:
        c = colourblindPalette20[parcel % len(colourblindPalette20)]
        if isinstance(c, str):
            rgb = np.array(mcolors.to_rgb(c))
        else:
            rgb = np.array(c[:3])
    else:
        rgb = np.array([0.7, 0.7, 0.7])
    return (rgb * 255).astype(np.uint8) if asUint8 else rgb


def majorityLabel(row):
    vals, counts = np.unique(row, return_counts=True)
    return vals[np.argmax(counts)]


def getBoundaryEdges(parcelFaces, allFaces, vertices):
    """Extract boundary edges of a set of faces (edges not shared with non-parcel faces)."""
    edgeCount = defaultdict(int)
    for face in parcelFaces:
        for i in range(3):
            edge = tuple(sorted([face[i], face[(i+1) % 3]]))
            edgeCount[edge] += 1

    allEdgeCount = defaultdict(int)
    for face in allFaces:
        for i in range(3):
            edge = tuple(sorted([face[i], face[(i+1) % 3]]))
            allEdgeCount[edge] += 1

    boundaryEdges = [
        edge for edge, count in edgeCount.items()
        if allEdgeCount[edge] < 2 or count == 1
    ]
    return boundaryEdges


def edgesToScatter3d(edges, vertices, colour, dash, name):
    """Convert edge list to a Scatter3d trace with gaps between edges."""
    xs, ys, zs = [], [], []
    for e in edges:
        xs += [vertices[e[0], 0], vertices[e[1], 0], None]
        ys += [vertices[e[0], 1], vertices[e[1], 1], None]
        zs += [vertices[e[0], 2], vertices[e[1], 2], None]
    return go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='lines',
        line=dict(color=colour, width=2, dash=dash),
        name=name,
        showlegend=False,
        hoverinfo='skip',
    )


def loadAgeMeshAndLabels(age, agePaths, ageKeyMap):
    """Load the atlas + cortical mesh for one age, sample parcel labels at each
    vertex (filling any unlabelled vertices via nearest-neighbour), and compute
    a per-face majority parcel label. Returns None if no paths are defined for
    this age.

    Returns a dict with: vertices, faces, labels (per-vertex), faceParcels
    (per-face), atlasParcelVals (set of non-zero parcel IDs in this atlas).
    """
    ageKey = ageKeyMap.get(age)
    if ageKey is None:
        print(f"No paths defined for age {age}, skipping.")
        return None

    paths = agePaths[ageKey]

    atlas_img = nib.load(str(paths['atlas']))
    atlas_data = atlas_img.get_fdata()
    atlas_affine = atlas_img.affine
    atlas_shape = atlas_data.shape

    gmvol_img = nib.load(str(paths['gmvol']))
    assert np.allclose(gmvol_img.affine, atlas_affine), f"WARNING: Atlas and GM affine mismatch for age {ageKey}!"

    mat = loadmat(str(paths['mat']))
    vertices = mat['vertices']
    faces = mat['faces'] - 1

    inv_affine = np.linalg.inv(atlas_affine)
    voxel_coords = nib.affines.apply_affine(inv_affine, vertices)
    voxel_coords = np.round(voxel_coords).astype(int)
    voxel_coords = np.clip(voxel_coords, [0, 0, 0], np.array(atlas_shape) - 1)

    labels = np.array([atlas_data[x, y, z] for x, y, z in voxel_coords])

    unlabelled_idx = np.where(labels == 0)[0]
    labelled_idx = np.where(labels != 0)[0]
    if len(unlabelled_idx) > 0:
        print(f"Age {ageKey}: filling {len(unlabelled_idx)} unlabelled vertices...")
        tree = cKDTree(vertices[labelled_idx])
        dists, nearest = tree.query(vertices[unlabelled_idx])
        labels[unlabelled_idx] = labels[labelled_idx][nearest]
        print("Label filling complete.")
    else:
        print(f"Age {ageKey}: no unlabelled vertices found.")

    faceLabels = labels[faces].astype(int)
    faceParcels = np.array([majorityLabel(row) for row in faceLabels])
    atlasParcelVals = set(np.unique(labels.astype(int))) - {0}

    return {
        'vertices': vertices,
        'faces': faces,
        'labels': labels,
        'faceParcels': faceParcels,
        'atlasParcelVals': atlasParcelVals,
    }


### Variant 1 — naive per-vertex colouring, all significant parcels

In [ ]:
for age in uniqueAges:
    meshData = loadAgeMeshAndLabels(age, agePaths, ageKeyMap)
    if meshData is None:
        continue
    vertices, faces, labels = meshData['vertices'], meshData['faces'], meshData['labels']

    uniqueParcelVals = np.unique(labels.astype(int))
    colourLookup = {
        parcel: parcelColour(parcel, age, validAgeCombos, asUint8=True)
        for parcel in uniqueParcelVals
        if parcel != 0
    }
    colourLookup[0] = np.array([128, 128, 128], dtype=np.uint8)
    vertexColours = np.array([colourLookup[int(label)] for label in labels], dtype=np.uint8)

    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    mesh.visual.vertex_colors = vertexColours

    fig = go.Figure(data=[
        go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            vertexcolor=[f'rgb({r},{g},{b})' for r, g, b in vertexColours[:, :3]],
            flatshading=False,
        )
    ])

    fig.update_layout(
        title=f"Age {age}: Significant parcels on cortical surface",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), bgcolor='white'),
        margin=dict(l=0, r=0, t=40, b=0), width=800, height=600,
    )

    significantParcels = set(parcel for (a, parcel, chrom) in validAgeCombos if a == age)
    for parcel in sorted(significantParcels):
        c = colourblindPalette20[parcel % len(colourblindPalette20)]
        r, g, b = [int(x * 255) for x in (mcolors.to_rgb(c) if isinstance(c, str) else c[:3])]
        fig.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None], mode='markers',
            marker=dict(size=10, color=f'rgb({r},{g},{b})'), name=str(parcel), showlegend=True,
        ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), bgcolor='white',
            camera=dict(eye=dict(x=0.9, y=0.9, z=0.9)),
        ),
        legend=dict(title='Parcel', bgcolor='white', bordercolor='lightgrey', borderwidth=1, itemsizing='constant'),
    )
    fig.show()


### Variant 2 — split significant/grey faces, transparent grey background

In [ ]:
def plotSplitMesh(age, meshData, validAgeCombos, greyColor='lightgrey', greyOpacity=0.2,
                   showHabOutlines=False, habGroups=None, cameraZoom=False):
    """Shared plotting routine for the 'split significant/grey faces' style,
    covering the original cell17/37/38 variants via parameters:
      - greyColor / greyOpacity: background mesh styling
      - showHabOutlines + habGroups: overlay habituation-category outlines
        (habGroups: list of (parcelSet, dashStyle, label) tuples, in precedence order)
      - cameraZoom: tighter camera framing
    """
    vertices, faces = meshData['vertices'], meshData['faces']
    faceParcels, atlasParcelVals = meshData['faceParcels'], meshData['atlasParcelVals']

    significantParcels = set(p for (a, p, c) in validAgeCombos if a == age)
    sigMask = np.array([fp in significantParcels for fp in faceParcels])
    sigFaces = faces[sigMask]
    greyFaces = faces[~sigMask]

    sigFaceColours = []
    for fp in faceParcels[sigMask]:
        c = colourblindPalette20[int(fp) % len(colourblindPalette20)]
        sigFaceColours.append(c if isinstance(c, str) else '#{:02x}{:02x}{:02x}'.format(c[0], c[1], c[2]))

    fig = go.Figure()

    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=greyFaces[:, 0], j=greyFaces[:, 1], k=greyFaces[:, 2],
        color=greyColor, opacity=greyOpacity, flatshading=False, showscale=False,
        lighting=dict(ambient=0.5, diffuse=0.9, specular=0.2, roughness=0.5),
        lightposition=dict(x=100, y=200, z=100), hoverinfo='skip',
    ))

    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=sigFaces[:, 0], j=sigFaces[:, 1], k=sigFaces[:, 2],
        facecolor=sigFaceColours, opacity=1.0, flatshading=False, showscale=False,
        lighting=dict(ambient=0.5, diffuse=0.8, specular=0.2, roughness=0.5),
        lightposition=dict(x=100, y=200, z=100),
    ))

    if showHabOutlines and habGroups:
        for habSet, dash, label in habGroups:
            habFaceIndices = []
            precedingSets = [s for s, _, _ in habGroups[:habGroups.index((habSet, dash, label))]]
            for fi, fp in enumerate(faceParcels):
                if (int(age), int(fp)) in habSet:
                    alreadyHandled = any((int(age), int(fp)) in higherSet for higherSet in precedingSets)
                    if not alreadyHandled:
                        habFaceIndices.append(fi)
            if len(habFaceIndices) == 0:
                continue
            habFaces = faces[habFaceIndices]
            boundaryEdges = getBoundaryEdges(habFaces, faces, vertices)
            if len(boundaryEdges) == 0:
                continue
            trace = edgesToScatter3d(boundaryEdges, vertices, 'black', dash, label)
            trace.showlegend = True
            fig.add_trace(trace)

    addedParcels = set()
    for parcel in sorted(significantParcels):
        if parcel in addedParcels or parcel not in atlasParcelVals:
            continue
        addedParcels.add(parcel)
        c = colourblindPalette20[int(parcel) % len(colourblindPalette20)]
        hexCol = c if isinstance(c, str) else '#{:02x}{:02x}{:02x}'.format(c[0], c[1], c[2])
        fig.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None], mode='markers',
            marker=dict(size=10, color=hexCol), name=str(parcel), showlegend=True,
        ))

    scene = dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), bgcolor='white')
    if cameraZoom:
        scene['camera'] = dict(eye=dict(x=0.9, y=0.9, z=0.9))

    fig.update_layout(
        title=f"Age {age}: Significant parcels on cortical surface",
        scene=scene,
        legend=dict(title='Parcel', bgcolor='white', bordercolor='lightgrey', borderwidth=1, itemsizing='constant'),
        margin=dict(l=0, r=0, t=40, b=0), width=900, height=700,
    )
    fig.show()


# Variant 2: transparent grey background, no habituation outlines (matches original cell 17)
for age in uniqueAges:
    meshData = loadAgeMeshAndLabels(age, agePaths, ageKeyMap)
    if meshData is None:
        continue
    plotSplitMesh(age, meshData, validAgeCombos, greyColor='lightgrey', greyOpacity=0.2)


### Variants 3 & 4 — split significant/grey faces with habituation-category outlines

Variant 3: opaque grey background (matches original cell 37). Variant 4: transparent grey background + tighter camera zoom (matches original cell 38).

In [ ]:
habParcels = set((int(row.Age), int(row.Parcel)) for _, row in adjustedHabDf.iterrows())
contHabParcels = set((int(row.Age), int(row.Parcel)) for _, row in adjustedDelHabDf.iterrows())
delHabParcels = set((int(row.Age), int(row.Parcel)) for _, row in adjustedLrHabDf.iterrows())

# Order of precedence: habParcels > contHabParcels > delHabParcels
habGroups = [
    (habParcels,     'solid', 'Habituation'),
    (contHabParcels, 'dash',  'Cont. Hab.'),
    (delHabParcels,  'dot',   'Ext. Hab.'),
]

# Variant 3: opaque grey background
for age in uniqueAges:
    meshData = loadAgeMeshAndLabels(age, agePaths, ageKeyMap)
    if meshData is None:
        continue
    plotSplitMesh(age, meshData, validAgeCombos, greyColor='rgb(128,128,128)', greyOpacity=1.0,
                  showHabOutlines=True, habGroups=habGroups, cameraZoom=False)

# Variant 4: transparent grey background, tighter camera zoom
for age in uniqueAges:
    meshData = loadAgeMeshAndLabels(age, agePaths, ageKeyMap)
    if meshData is None:
        continue
    plotSplitMesh(age, meshData, validAgeCombos, greyColor='lightgrey', greyOpacity=0.2,
                  showHabOutlines=True, habGroups=habGroups, cameraZoom=True)


## Save group-averaged HRFs for downstream analysis

Trial ranges per epoch (Fam1 = trials 1-5, Fam2 = 6-10, Fam3 = 11-15, Novel = 16-20, Post = 21-25).

In [ ]:
EPOCH_TRIAL_RANGES = {
    'Fam1':  (1,  5),
    'Fam2':  (6,  10),
    'Fam3':  (11, 15),
    'Novel': (16, 20),
    'Post':  (21, 25),
}


def _pad_or_truncate(ts, target_len):
    if len(ts) > target_len:
        return ts[:target_len]
    if len(ts) < target_len:
        return np.pad(ts, (0, target_len - len(ts)), mode='constant', constant_values=np.nan)
    return ts


hrf_rows = []

for age in sorted(metadataDf['Age'].unique()):
    age_meta = metadataDf[metadataDf['Age'] == age]

    for epoch_name, (t_lo, t_hi) in EPOCH_TRIAL_RANGES.items():
        epoch_meta = age_meta[(age_meta['Trial'] >= t_lo) & (age_meta['Trial'] <= t_hi)]

        for (parcel, chrom), combo_df in epoch_meta.groupby(['Parcel', 'Chrom']):
            infant_curves = []
            for infant, infant_df in combo_df.groupby('Infant'):
                trial_curves = [
                    _pad_or_truncate(tsDataset[idx] * 1e6, targetLen)
                    for idx in infant_df['tsIDx']
                    if not np.isnan(tsDataset[idx]).all()
                ]
                if trial_curves:
                    infant_curves.append(np.nanmean(trial_curves, axis=0))

            if not infant_curves:
                continue

            stack = np.vstack(infant_curves)
            n_inf = stack.shape[0]
            mean_curve = np.nanmean(stack, axis=0)
            sem_curve = np.nanstd(stack, axis=0) / np.sqrt(n_inf)

            for s in range(targetLen):
                hrf_rows.append({
                    'Age': int(age), 'Parcel': int(parcel), 'Chrom': int(chrom),
                    'Epoch': epoch_name, 'sample_idx': s,
                    'mean': float(mean_curve[s]), 'sem': float(sem_curve[s]),
                    'n_infants': n_inf,
                })

groupHrfDf = pd.DataFrame(hrf_rows)

hrf_out_path = tablesDir / 'group_averaged_HRFs_by_epoch.csv'
groupHrfDf.to_csv(hrf_out_path, index=False)

print(f"Saved group HRFs: {hrf_out_path}")
print(f"  Rows: {len(groupHrfDf):,}")
print(f"  Unique (Age, Parcel, Chrom, Epoch) combos: {groupHrfDf.groupby(['Age','Parcel','Chrom','Epoch']).ngroups:,}")
print(f"  n_infants summary per Age x Epoch:")
print(groupHrfDf.groupby(['Age','Epoch'])['n_infants'].agg(['median','min','max']).to_string())
